# Realizing an estimand

`realize(estimand, producer)` turns a declaration into a number with its provenance. The
producer is anything satisfying `SupportsEstimands` — here a fitted surface. Three things
are checked before any arithmetic: the producer's **capabilities** (an estimand needing one
the producer lacks is `Unsupported`), the **identification verdict** (a blocked or downgraded
route is refused unless you `assume_identified=True`, which writes a ledger line), and the
**dimension** (derived from the producer, asserted against the declaration). Units are
converted with a ledger line when the estimand is declared in a different unit of the same
dimension.

In [ ]:
import numpy as np

from axiom.core import UNITS, Capability, D, Intervention, Population, Posterior, TimeWindow, Treatment, Verdict, is_failure
from axiom.estimands import (
    Estimand, EstimandRegistry, EstimandResult, Level, Quantity, RealizedDraws, ResultStatus,
    check_estimand_dimension, estimand_expr, evaluate, realize, standard_estimands, substitute,
)
from axiom.identify import CausalGraph, identify
from axiom.sim import arms_world, surface_world
from axiom.surface import HillKernel, fit

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

## A fitted producer

In [ ]:
world = arms_world(n_units=30, treatments=("dose",), kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
                   truth={"beta_dose": 10.0, "k_dose": 50.0, "s_dose": 2.0, "alpha": 1.0}, seed=0)
res = fit(world.spec, world.panel, backend="laplace", draws=2000, seed=0)
print(res.converged, sorted(c.value for c in res.capabilities()))
print(res.treatments[0].name, res.outcome_unit, res.dose_unit("dose"), res.n_units, res.n_periods)

## Declare, identify, realize

The standard registry builds the domain-general library for one treatment/outcome pair. The
graph for a randomized arms world is `dose -> y` with nothing else, so the route is identified.

In [ ]:
spec = world.spec
registry: EstimandRegistry = standard_estimands(
    treatment=spec.treatments[0], outcome=spec.outcome, population=Population(name="all"),
    window=TimeWindow(start=0, stop=1), level=Level(unit="individual"), dose=60.0, reference_dose=0.0,
)
print(registry.names())
verdict = identify(CausalGraph.from_edges("dose -> y"), "dose", "y").verdict
print(verdict.status, verdict.route)

In [ ]:
results = evaluate(list(registry), res, verdicts={name: verdict for name in registry.names()}, definition="hdi", mass=0.9)
rows = []
for name, r in results.items():
    if isinstance(r, EstimandResult):
        rows.append(
            [name, r.status, f"{r.summary.mean:.4f}", str(r.summary.interval), str(r.dimension), r.unit]
        )
    else:
        rows.append([name, str(r), "", "", "", ""])
table(rows, headers=("estimand", "status", "mean", "interval", "dimension", "unit"))

Against the world's truth: the contrast is the true response difference, the marginal is the
closed-form derivative, and every number above carries its interval definition and mass.

In [ ]:
truth_contrast = float(world.forward({"dose": 60.0}).mean() - world.forward({"dose": 0.0}).mean())
truth_marginal = float(np.asarray(world.marginal("dose", dose={"dose": 60.0})).mean())
print("truth contrast", round(truth_contrast, 4), "| truth marginal", round(truth_marginal, 5))

## Honest refusals

A producer stripped of a capability returns `Unsupported`; a blocked verdict returns `Blocked`
unless the analyst asserts identification — and that assertion is a ledger line, not a flag
that disappears.

In [ ]:
lift = registry.get("contrast_at_dose")
crippled = res.restrict([Capability.COUNTERFACTUAL])
print(realize(lift, crippled))

blocked = identify(CausalGraph.from_edges("dose -> y, dose <-> y"), "dose", "y").verdict
print(realize(lift, res, verdict=blocked))
asserted = realize(lift, res, verdict=blocked, assume_identified=True)
if isinstance(asserted, EstimandResult):
    status: ResultStatus = asserted.status
    print(status, [line.kind for line in asserted.ledger], [a.name for a in asserted.assumptions])

## Units cross the boundary with a ledger line

Declare the same estimand in cents per gram when the fit was in USD and kg: the realized value
is converted, the dose level is interpreted in the estimand's unit, and the ledger says so.

In [ ]:
for unit, base in (("USD", "currency"), ("cent", "currency"), ("kg", "outcome"), ("g", "outcome")):
    UNITS.declare(unit, base)
UNITS.register("USD", "cent", 100)
UNITS.register("kg", "g", 1000)
in_cents = lift.model_copy(update={
    "treatment": Treatment(name="dose", dimension=D.currency, unit="cent"),
    "intervention": Intervention(doses={"dose": 6000.0}),
    "reference": Intervention(doses={"dose": 0.0}),
    "outcome": spec.outcome.model_copy(update={"unit": "g"}),
})
converted = realize(in_cents, res, verdict=verdict)
base_result = results["contrast_at_dose"]
if isinstance(converted, EstimandResult) and isinstance(base_result, EstimandResult):
    print(round(base_result.summary.mean, 4), "kg  ->", round(converted.summary.mean, 2), converted.unit)
    for line in converted.ledger:
        print("  ledger:", line.statement)

## Keeping the draws, and the estimand as an expression

`keep_draws=True` returns the draws alongside the result. `estimand_expr` writes the estimand
as an expression over the model's mean tree — `substitute` pins the treatment's data node to
the intervention's dose — so the dimension gate can check every declared estimand symbolically.

In [ ]:
rd = realize(lift, res, verdict=verdict, keep_draws=True)
if isinstance(rd, RealizedDraws):
    print(rd.draws.shape, rd.result.summary.n)

from axiom.core import Data, dimension
tree = estimand_expr(lift, mean=world.model.mean, treatment_data=Data(name="dose", dimension=D.currency))
if not is_failure(tree):
    print(dimension(tree), "==", check_estimand_dimension(lift, tree))
pinned = substitute(world.model.mean, {"dose": __import__("axiom.core", fromlist=["Const"]).Const(value=60.0, dimension=D.currency)})
print(type(pinned).__name__)

## Where the facets meet the panel

On a panel world the `window` (basis) and `level` facets are arithmetic on the realized draws:
a cumulative contrast over a window sums periods; a per-period one averages them; an
`individual` level averages units (with optional weights), `aggregate` sums them.

In [ ]:
pw = surface_world(n_units=4, n_periods=20, treatments=("a",), kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
                   truth={"beta_a": 10.0, "k_a": 50.0, "s_a": 2.0}, intercept="shared", seed=1)
pres = fit(pw.spec, pw.panel, backend="laplace", draws=1000, seed=0)
base = dict(quantity=Quantity(kind="contrast"), treatment=pw.spec.treatments[0], intervention=Intervention(doses={"a": 80.0}),
            reference=Intervention(doses={"a": 0.0}), outcome=pw.spec.outcome, population=Population(name="all"), dimension=pw.spec.outcome.dim)
for label, window, level in (("cumulative/aggregate", TimeWindow(start=0, stop=20, basis="cumulative"), Level(unit="aggregate")),
                             ("per_period/individual", TimeWindow(start=0, stop=20, basis="per_period"), Level(unit="individual")),
                             ("window [5,10)/individual", TimeWindow(start=5, stop=10, basis="per_period"), Level(unit="individual"))):
    e = Estimand(name="lift", window=window, level=level, **base)
    r = realize(e, pres, verdict=verdict)
    print(f"{label:26s}", r if is_failure(r) else f"{r.summary.mean:9.3f} {r.summary.interval}")

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.core import D, Intervention, Outcome, Population, TimeWindow, Treatment
from axiom.display import show
from axiom.estimands import Estimand, Level, Quantity
from axiom.viz import transfer

fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
shared = dict(
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=Outcome(name="yield_total", dimension=D.outcome, unit="kg"),
    window=TimeWindow(start=0, stop=8),
    level=Level(unit="cluster"),
    dimension=D.outcome,
)
here = Estimand(name="trial", population=Population(name="north"), **shared)
there = Estimand(name="region", population=Population(name="whole_region"), **shared)

plan = here.transfer_to(there)
show(plan)
transfer(plan)

No numeric correction was required here: the two populations differ, and the entire cost of the move is one assumption — `s_admissibility` — that someone now has to defend. That is the reading worth taking away. A transfer can leave the number completely untouched and still change what the number means, which is why the plan reports the assumption and the ledger line and not just a corrected estimate.